<a href="https://colab.research.google.com/github/matiasnr96/cloud-provider-analytics/blob/main/notebooks/01_exploracion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

LANDING_PATH = Path("/content/drive/MyDrive/cloud-provider-analytics-data/landing")

print(LANDING_PATH)

/content/drive/MyDrive/cloud-provider-analytics-data/landing


In [3]:
import pandas as pd

customers = pd.read_csv(LANDING_PATH / "customers_orgs.csv")

customers.head()

,org_id,org_name,industry,hq_region,plan_tier,is_enterprise,signup_date,sales_rep,lifecycle_stage,marketing_source,nps_score
0,org_xaji0y6d,Nimbus Labs 0,Education,sa-east,standard,True,2025-05-26,rep_c,churned,partner,NaN
1,org_pbhsahxt,Nova Tech 1,Education,us-east,standard,False,2025-06-16,rep_d,lead,event,-3.0
2,org_hv3a3zmf,Gamma Data 2,Media,eu-central,pro,True,2025-05-12,rep_c,active,organic,4.0
3,org_8mdd4v30,Apex Data 3,Manufacturing,sa-east,standard,False,2025-05-30,rep_b,active,ads,12.0
4,org_t9nt3w5u,Delta Digital 4,E-commerce,us-west,standard,False,2025-05-16,rep_d,active,partner,17.0


## 1. Customers / Organizations

El archivo `customers_orgs.csv` contiene información de las organizaciones o clientes del proveedor cloud.

En esta primera exploración se analizarán:
- dimensiones del dataset;
- columnas disponibles;
- tipos de datos;
- valores nulos;
- valores únicos;
- posibles problemas de calidad.

In [4]:
print("Cantidad de filas:", customers.shape[0])
print("Cantidad de columnas:", customers.shape[1])

Cantidad de filas: 80
Cantidad de columnas: 11


In [5]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   org_id            80 non-null     object 
 1   org_name          80 non-null     object 
 2   industry          80 non-null     object 
 3   hq_region         80 non-null     object 
 4   plan_tier         80 non-null     object 
 5   is_enterprise     80 non-null     bool   
 6   signup_date       80 non-null     object 
 7   sales_rep         80 non-null     object 
 8   lifecycle_stage   80 non-null     object 
 9   marketing_source  80 non-null     object 
 10  nps_score         69 non-null     float64
dtypes: bool(1), float64(1), object(9)
memory usage: 6.5+ KB


In [6]:
customers.dtypes

,0
org_id,object
org_name,object
industry,object
hq_region,object
plan_tier,object
is_enterprise,bool
signup_date,object
sales_rep,object
lifecycle_stage,object
marketing_source,object


In [7]:
customers.isnull().sum()

,0
org_id,0
org_name,0
industry,0
hq_region,0
plan_tier,0
is_enterprise,0
signup_date,0
sales_rep,0
lifecycle_stage,0
marketing_source,0


In [8]:
print("Filas duplicadas:", customers.duplicated().sum())
print("org_id duplicados:", customers["org_id"].duplicated().sum())

Filas duplicadas: 0
org_id duplicados: 0


In [9]:
customers["org_id"].duplicated()

,org_id
0,False
1,False
2,False
3,False
4,False
...,...
75,False
76,False
77,False
78,False


In [10]:
print("Filas duplicadas:", customers.duplicated().sum())
print("org_id duplicados:", customers["org_id"].duplicated().sum())

Filas duplicadas: 0
org_id duplicados: 0


In [11]:
print("NPS mínimo:", customers["nps_score"].min())
print("NPS máximo:", customers["nps_score"].max())

nps_fuera_rango = customers[
    (customers["nps_score"] < 0) |
    (customers["nps_score"] > 10)
]

print("NPS fuera del rango 0-10:", len(nps_fuera_rango))

NPS mínimo: -38.0
NPS máximo: 101.0
NPS fuera del rango 0-10: 59


In [12]:
nps_fuera_rango[["org_id", "org_name", "nps_score"]]

,org_id,org_name,nps_score
1,org_pbhsahxt,Nova Tech 1,-3.0
3,org_8mdd4v30,Apex Data 3,12.0
4,org_t9nt3w5u,Delta Digital 4,17.0
6,org_wnnhj7xv,Alpha Data 6,14.0
8,org_41ibljh7,Gamma Cloud 8,22.0
9,org_5lxo6qji,Omega Labs 9,12.0
10,org_ujv6oh9s,Delta Digital 10,51.0
11,org_dbdw2pcn,Nimbus AI 11,12.0
13,org_jxepq85j,Alpha Cloud 13,81.0
15,org_1t2tala7,Gamma Data 15,41.0


### Conclusiones de `customers_orgs.csv`

El archivo tiene 80 organizaciones y 11 columnas. Cada fila representa una organización distinta, ya que no encontramos `org_id` repetidos ni filas duplicadas.

En general, los datos están bastante completos. El principal problema aparece en `nps_score`: hay 11 valores nulos y 59 valores que están fuera del rango esperado de 0 a 10. El mínimo encontrado fue -38 y el máximo 101, por lo que claramente hay valores que tendremos que revisar más adelante.

También vimos que `signup_date` fue leído como texto, así que durante el procesamiento habrá que convertirlo a formato fecha.

Por ahora no vamos a modificar estos datos, ya que estamos trabajando con la información original de la capa Landing. En esta etapa solamente buscamos conocer los datos y detectar posibles problemas de calidad.

In [14]:
users = pd.read_csv(LANDING_PATH / "users.csv")

users.head()

,user_id,org_id,email,role,active,created_at,last_login
0,user_9tze5r5u,org_3t60rjiw,user_9tze5r5u@example.com,devops,True,2025-07-10,NaN
1,user_qpgb7r3o,org_afeyuhz1,user_qpgb7r3o@example.com,developer,True,2025-07-15,2025-08-10
2,user_cwbfuk9e,org_ykl1cq99,user_cwbfuk9e@example.com,developer,True,2025-06-12,2025-07-26
3,user_1v2isqp4,org_g8sbi4q2,user_1v2isqp4@example.com,developer,False,2025-05-12,NaN
4,user_9kwv08hh,org_pht0hl9x,user_9kwv08hh@example.com,ml_engineer,True,2025-08-01,2025-08-11


In [15]:
print("Cantidad de filas:", users.shape[0])
print("Cantidad de columnas:", users.shape[1])

users.info()

Cantidad de filas: 800
Cantidad de columnas: 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     800 non-null    object
 1   org_id      800 non-null    object
 2   email       800 non-null    object
 3   role        800 non-null    object
 4   active      800 non-null    bool  
 5   created_at  800 non-null    object
 6   last_login  661 non-null    object
dtypes: bool(1), object(6)
memory usage: 38.4+ KB


In [16]:
print("Filas duplicadas:", users.duplicated().sum())
print("user_id duplicados:", users["user_id"].duplicated().sum())

Filas duplicadas: 0
user_id duplicados: 0


In [17]:
org_ids_validos = set(customers["org_id"])

usuarios_sin_org = users[
    ~users["org_id"].isin(org_ids_validos)
]

print("Usuarios con org_id inexistente:", len(usuarios_sin_org))

Usuarios con org_id inexistente: 0


### Conclusiones de `users.csv`

El archivo contiene 800 usuarios y 7 columnas. No se encontraron usuarios ni filas duplicadas.

La única columna con datos faltantes es `last_login`, con 139 valores nulos. Además, `created_at` y `last_login` fueron leídos como texto y más adelante deberán convertirse a fecha.

Todos los usuarios tienen un `org_id` válido, por lo que no se encontraron problemas en la relación con las organizaciones.

In [18]:
resources = pd.read_csv(LANDING_PATH / "resources.csv")

resources.head()

,resource_id,org_id,service,region,created_at,state,tags_json
0,res_eubfn9kr,org_pnsm43d8,compute,sa-east,2025-08-14,running,"[""env:prod""]"
1,res_fvb66h3r,org_i7p5tb94,database,ap-south,2025-06-05,stopped,NaN
2,res_cbrlqmn4,org_d14ve92m,storage,eu-central,2025-08-10,running,"[""env:prod"", ""pii:true""]"
3,res_ew1yf0dw,org_pja1wj0t,networking,us-west,2025-06-02,running,"[""pii:true""]"
4,res_n6mbypjd,org_pja1wj0t,storage,us-west,2025-06-05,running,"[""env:prod"", ""costcenter:beta""]"


In [19]:
print("Cantidad de filas:", resources.shape[0])
print("Cantidad de columnas:", resources.shape[1])

resources.info()

Cantidad de filas: 400
Cantidad de columnas: 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resource_id  400 non-null    object
 1   org_id       400 non-null    object
 2   service      400 non-null    object
 3   region       400 non-null    object
 4   created_at   400 non-null    object
 5   state        400 non-null    object
 6   tags_json    317 non-null    object
dtypes: object(7)
memory usage: 22.0+ KB


In [20]:
resources.isnull().sum()

,0
resource_id,0
org_id,0
service,0
region,0
created_at,0
state,0
tags_json,83


In [21]:
print("Filas duplicadas:", resources.duplicated().sum())
print("resource_id duplicados:", resources["resource_id"].duplicated().sum())

Filas duplicadas: 0
resource_id duplicados: 0


In [22]:
recursos_sin_org = resources[
    ~resources["org_id"].isin(customers["org_id"])
]

print("Recursos con org_id inexistente:", len(recursos_sin_org))

Recursos con org_id inexistente: 0


In [23]:
resources["service"].value_counts()

,count
service,
compute,116
storage,71
database,68
networking,64
analytics,42
genai,39


### Conclusiones de `resources.csv`

El archivo contiene 400 recursos y no se encontraron registros ni `resource_id` duplicados.

Todos los recursos están asociados a una organización existente. Se encontraron 6 tipos de servicios diferentes y el único campo con datos faltantes es `tags_json`, con 83 valores nulos.

## 4. Exploración general de las fuentes restantes

A continuación se realiza una revisión general de los archivos restantes para conocer su tamaño, valores faltantes y posibles registros duplicados.

In [24]:
archivos = [
    "support_tickets.csv",
    "marketing_touches.csv",
    "nps_surveys.csv",
    "billing_monthly.csv"
]

for archivo in archivos:
    df = pd.read_csv(LANDING_PATH / archivo)

    print(f"\n--- {archivo} ---")
    print("Filas:", df.shape[0])
    print("Columnas:", df.shape[1])
    print("Filas duplicadas:", df.duplicated().sum())
    print("\nValores nulos:")
    print(df.isnull().sum())


--- support_tickets.csv ---
Filas: 1000
Columnas: 8
Filas duplicadas: 0

Valores nulos:
ticket_id         0
org_id            0
category          0
severity          0
created_at        0
resolved_at     240
csat            254
sla_breached      0
dtype: int64

--- marketing_touches.csv ---
Filas: 1500
Columnas: 7
Filas duplicadas: 0

Valores nulos:
touch_id     0
org_id       0
campaign     0
channel      0
timestamp    0
clicked      0
converted    0
dtype: int64

--- nps_surveys.csv ---
Filas: 92
Columnas: 4
Filas duplicadas: 0

Valores nulos:
org_id          0
survey_date     0
nps_score      19
comment        10
dtype: int64

--- billing_monthly.csv ---
Filas: 240
Columnas: 8
Filas duplicadas: 0

Valores nulos:
invoice_id                0
org_id                    0
month                     0
subtotal                  0
credits                 137
taxes                     0
currency                  0
exchange_rate_to_usd      0
dtype: int64


### Conclusiones de las fuentes restantes

No se encontraron filas duplicadas ni organizaciones inexistentes en ninguno de los cuatro archivos analizados.

Los principales faltantes aparecen en `support_tickets.csv`, en las columnas `resolved_at` y `csat`. `nps_surveys.csv` también presenta algunos valores faltantes en NPS y comentarios, mientras que `billing_monthly.csv` tiene varios valores nulos en `credits`.

`marketing_touches.csv` no presentó valores faltantes.

## 5. Usage Events Stream

Esta fuente contiene eventos de uso de los servicios cloud distribuidos en varios archivos JSONL. Vamos a revisar la cantidad de archivos y eventos, además de observar su estructura inicial.

In [26]:
STREAM_PATH = LANDING_PATH / "usage_events_stream"

archivos_stream = list(STREAM_PATH.glob("*.jsonl"))

print("Cantidad de archivos JSONL:", len(archivos_stream))

Cantidad de archivos JSONL: 120


In [27]:
total_eventos = 0

for archivo in archivos_stream:
    with open(archivo, "r") as f:
        total_eventos += sum(1 for linea in f if linea.strip())

print("Cantidad total de eventos:", total_eventos)

Cantidad total de eventos: 43200


In [28]:
ejemplo_stream = pd.read_json(archivos_stream[0], lines=True)

print("Archivo analizado:", archivos_stream[0].name)
print("Cantidad de registros:", ejemplo_stream.shape[0])
print("Cantidad de columnas:", ejemplo_stream.shape[1])

ejemplo_stream.head()

Archivo analizado: events_part_0069.jsonl
Cantidad de registros: 360
Cantidad de columnas: 13


,event_id,timestamp,org_id,resource_id,service,region,metric,value,unit,cost_usd_increment,schema_version,carbon_kg,genai_tokens
0,evt_c5rhcrkjh59e,2025-08-01 22:49:00+00:00,org_axeqbnhl,res_nd7g0l3r,analytics,eu-central,storage_gb_hours,14.9331,gb_hours,1.1239,2,0.002987,NaN
1,evt_hrqcdkf8icwj,2025-07-12 11:34:00+00:00,org_t1ghr09s,res_k38vhmew,compute,ap-south,storage_gb_hours,13.1295,gb_hours,1.2313,1,NaN,NaN
2,evt_l5ws8s604cko,2025-08-27 05:12:00+00:00,org_t9nt3w5u,res_71v4kfiw,compute,us-east,requests,104.0000,None,10.6258,2,0.020800,NaN
3,evt_3gc4zre0xgmt,2025-08-18 23:25:00+00:00,org_1swjckjl,res_3lww7opi,compute,ap-south,requests,110.0000,count,9.7306,2,0.022000,NaN
4,evt_v2ntznddiaw3,2025-07-28 08:51:00+00:00,org_1t2tala7,res_aw1lxxd8,compute,eu-central,storage_gb_hours,14.6657,gb_hours,1.3278,2,0.002933,NaN


In [29]:
eventos = pd.concat(
    [pd.read_json(archivo, lines=True) for archivo in archivos_stream],
    ignore_index=True
)

print("Total de eventos:", len(eventos))
print("Cantidad de columnas:", eventos.shape[1])

Total de eventos: 43200
Cantidad de columnas: 13


In [30]:
print("Versiones de esquema:")
print(eventos["schema_version"].value_counts().sort_index())

print("\nValores nulos:")
print(eventos.isnull().sum())

Versiones de esquema:
schema_version
1    10800
2    32400
Name: count, dtype: int64

Valores nulos:
event_id                  0
timestamp                 0
org_id                    0
resource_id               0
service                   0
region                    0
metric                    0
value                   877
unit                   2075
cost_usd_increment        0
schema_version            0
carbon_kg             10800
genai_tokens          40068
dtype: int64


In [31]:
print("Eventos duplicados:", eventos.duplicated().sum())
print("event_id duplicados:", eventos["event_id"].duplicated().sum())

print(
    "Costos negativos:",
    (eventos["cost_usd_increment"] < 0).sum()
)

Eventos duplicados: 0
event_id duplicados: 0
Costos negativos: 216


### Conclusiones de `usage_events_stream`

La fuente contiene 43.200 eventos distribuidos en 120 archivos JSONL. No se encontraron eventos ni identificadores duplicados.

Se encontraron dos versiones de esquema: 10.800 eventos con versión 1 y 32.400 con versión 2. También aparecen algunos problemas de calidad, como 877 valores nulos en `value`, 2.075 en `unit` y 216 costos negativos.

Las diferencias en `carbon_kg` y `genai_tokens` están relacionadas con los cambios de esquema y deberán contemplarse más adelante durante el procesamiento.